# Step 6 -- Directory Dashboard (interactive)

Builds the end-user dashboard from whatever Step 5 has produced so far. Rerun it after
processing more pages, or after adding another edition to `DATASETS`, and it rebuilds from
what is on disk.

**Outputs** (into `dashboard_output/`)

| File | What it is |
|---|---|
| `Directory_Dashboard.html` | Single self-contained file. Browse + interactive Overview. No internet, no server, no libraries. |
| `Directory_Dashboard.xlsx` | Same data plus relationship counts and the review flags. |

**What is in it**

- **Browse** -- search, filter, sortable columns, click an entry for its full record and its
  connections to other entries. Type and Marker are checkbox multi-selects (a slide-down panel
  under the button), so you can filter to e.g. "person + business" or several specific markers
  at once, not just one value at a time.
- **Overview -- At a glance** -- four small charts (entry type, housing type, top occupations,
  top streets) always visible together over the whole dataset, so the shape of the data reads
  in one look, the way a static figure would.
- **Overview -- Explore a metric** -- the same idea, but interactive: pick the metric, the
  ordering, how many bars, whether to split by marker, and whether the chart follows the Browse
  filters. Click any bar, in either section, to filter the table to those people.
- **Marker column**, exactly as printed and blank where the directory printed nothing.
- **Needs-review flag**, split into a fragment signal and a duplicate signal, reported and
  counted separately (see section 6 -- the duplicate check in particular is a judgment call,
  not a fact, and section 6 explains why).
- **Abbreviations panel**, because `bds`, `wks` and `(c)` mean nothing to a first-time reader.
- **Download CSV** of whatever is currently filtered, and of the chart data.
- **Sounds-like search**, so an OCR slip like *Kbell* for *Abell* still turns up.
- **Permalinks** -- opening `...html#entry-2841` jumps straight to that record, so a specific
  entry can be cited and shared.

## 1. Configuration

In [1]:
import difflib
import json
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd
import openpyxl
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

DATASETS = [
    {"label": "1900-1901", "output_dir": Path("step5_new_pages_output")},
]
DASHBOARD_DIR = Path("dashboard_output")

# similarity, on the full raw text of two neighbouring same-page entries,
# above which they are called a probable duplicate. Checked empirically
# against every adjacent pair in the 5,621-entry dataset (see notebook
# markdown for the full walkthrough): this directory is alphabetically
# sorted and dense with real siblings, parent/child pairs, and common
# surnames at a shared address, so similar-looking neighbours are common
# and legitimate -- ratios up to 0.85 are dominated by genuinely different
# people (e.g. "Anderson John (c)" appears twice as two different men).
# 0.92 is the point where OCR-noise-only differences (one or two swapped
# letters, e.g. "Boule Emery" / "Boyle Emery" at the same address and job)
# separate out from real-but-similar family entries. There is no threshold
# that perfectly separates the two; this one trades recall for precision
# deliberately, so the flag stays a credible "worth a look," not noise.
DUP_THRESHOLD = 0.92

## 2. The abbreviations glossary

Powers the Abbreviations panel and the hover text on entries. The directory abbreviates nearly
everything, and the transcription deliberately preserves those abbreviations rather than
expanding them, so the digitized record matches the page. That makes a glossary necessary for
anybody reading it who is not already familiar with the source.

In [2]:
GLOSSARY = {
    "groups": [
        {"name": "Where someone lived",
         "items": ["r.", "h.", "bds", "rms", "res", "hhldr", "same", "cor", "bt", "ns", "ss", "es", "ws", "blk", "al", "ave", "jct", "Hts", "add"]},
        {"name": "Common occupations",
         "items": ["lab", "clk", "wks", "carp", "servt", "engr", "bkpr", "machst", "dressmkr",
                   "trav slsmn", "drayman", "teamster", "condr", "slsmn", "mngr", "propr",
                   "prest", "treasr", "secy", "supt", "insp", "hlpr", "opr", "blksmith", "fctv"]},
        {"name": "Markers and personal notes",
         "items": ["(c)", "(col)", "wid", "Miss", "Mrs.", "jr.", "sr."]},
        {"name": "Railroads and large employers",
         "items": ["S. P.", "H. & T. C.", "G. H. & S. A.", "T. & N. O.", "H. E. & W. T.", "I. & G. N.", "frt", "yds", "shops"]},
    ],
    "terms": {
        "r.": "residence -- where the person lived",
        "h.": "house -- lived there and owned the home",
        "bds": "boards -- lodged at this address, meals included",
        "rms": "rooms -- rented a room at this address",
        "res": "residence",
        "hhldr": "householder -- head of the household",
        "same": "same address as the entry above, or as the employer just named",
        "cor": "corner of",
        "bt": "between",
        "ns": "north side of", "ss": "south side of", "es": "east side of", "ws": "west side of",
        "blk": "block", "al": "alley", "ave": "avenue", "jct": "junction",
        "Hts": "Heights (Houston Heights)", "add": "addition -- a named subdivision",
        "lab": "laborer", "clk": "clerk", "wks": "works for or at",
        "carp": "carpenter", "servt": "servant", "engr": "engineer",
        "bkpr": "bookkeeper", "machst": "machinist", "dressmkr": "dressmaker",
        "trav slsmn": "travelling salesman", "drayman": "driver of a dray, a heavy cart",
        "teamster": "driver of a team of horses", "condr": "conductor",
        "slsmn": "salesman", "propr": "proprietor", "prest": "president",
        "treasr": "treasurer", "secy": "secretary", "supt": "superintendent",
        "insp": "inspector", "hlpr": "helper", "opr": "operator",
        "blksmith": "blacksmith", "fctv": "factory", "mngr": "manager",
        "(c)": "the directory's marker for a Black resident. The publisher marked only "
               "non-white residents, so an unmarked entry was the convention for white residents",
        "(col)": "another form of the same marker",
        "wid": "widow of the name that follows",
        "Miss": "unmarried woman", "Mrs.": "married woman",
        "jr.": "junior", "sr.": "senior",
        "S. P.": "Southern Pacific railroad",
        "H. & T. C.": "Houston & Texas Central railroad",
        "G. H. & S. A.": "Galveston, Harrisburg & San Antonio railway",
        "T. & N. O.": "Texas & New Orleans railroad",
        "H. E. & W. T.": "Houston East & West Texas railway",
        "I. & G. N.": "International & Great Northern railroad",
        "frt": "freight", "yds": "yards", "shops": "railroad repair shops",
    },
}

## 3. Loading Step 5 output

Prefers the combined CSV, falls back to per-page CSVs, and tolerates pages that legitimately
produced no entries (full-page advertisements and dividers).

In [3]:
# ---------------------------------------------------------------- loading
def load_edition(label, output_dir):
    combined = output_dir / "csv" / "all_pages_combined.csv"
    if combined.exists():
        try:
            df = pd.read_csv(combined)
        except pd.errors.EmptyDataError:
            df = pd.DataFrame()
    else:
        frames = []
        for p in sorted((output_dir / "csv").glob("*.csv")):
            if p.name == "all_pages_combined.csv":
                continue
            try:
                frames.append(pd.read_csv(p))
            except pd.errors.EmptyDataError:
                continue
        df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    df["edition"] = label
    return df


def load_all(datasets=DATASETS):
    frames = []
    for ds in datasets:
        if not ds["output_dir"].exists():
            print(f"  skipping {ds['label']}: {ds['output_dir']} not found")
            continue
        df = load_edition(ds["label"], ds["output_dir"])
        print(f"  {ds['label']}: {len(df)} entries")
        frames.append(df)
    if not frames:
        raise SystemExit("No data found -- check DATASETS.")
    df = pd.concat(frames, ignore_index=True).reset_index(drop=True)
    df["id"] = df.index
    return df

## 4. Text normalisation

In [4]:
# ---------------------------------------------------------- normalisation
def norm(s):
    """Trimmed string, with pandas NaN treated as blank. NaN is truthy, so
    `str(s or "")` would produce the literal text 'nan' and group every blank
    field together under that fake key."""
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    s = str(s)
    return "" if s.strip().lower() == "nan" else re.sub(r"\s+", " ", s).strip()


def nkey(s):
    return re.sub(r"[^a-z]", "", norm(s).lower())


ADDR_PREFIX_RE = re.compile(r"^(r\.?|h\.?|bds\.?|rms\.?|res\.?|residence|rooms)\s+", re.I)
NON_BUSINESS_PAREN_RE = re.compile(r"^(c|col|colored|wid\b.*|mrs\.?\s.*|miss\b.*|jr\.?|sr\.?|inc\.?)$", re.I)
PAREN_RE = re.compile(r"\(([^)]+)\)")
DIRWORD_RE = re.compile(r"^\s*((n|s|e|w|ne|nw|se|sw)\s*(s|w|e)?\s+|cor\s+|rear\s+|over\s+|opp\s+|near\s+)", re.I)
HOUSENUM_RE = re.compile(r"^\s*[\d½¼¾\-\s/]+")
PLACEHOLDER_ADDR = {"same", "do", "ditto"}


def normalize_address(*fields):
    for f in fields:
        f = norm(f)
        if not f:
            continue
        f = ADDR_PREFIX_RE.sub("", f).rstrip(".").strip().lower()
        if f and f not in PLACEHOLDER_ADDR:
            return f
    return None


def normalize_business(s):
    s = norm(s).lower()
    s = re.sub(r"[.,]", "", s)
    s = re.sub(r"\b(the|inc|co)\b", "", s)
    return re.sub(r"\s+", " ", s).strip() or None


def extract_paren_affiliation(raw):
    m = PAREN_RE.search(raw or "")
    if not m:
        return None
    c = m.group(1).strip()
    return None if (NON_BUSINESS_PAREN_RE.match(c) or len(c) < 4) else c


def street_name(*fields):
    """Street from an address. Original capitalisation is preserved, because
    .title() would turn McKinney into Mckinney."""
    for f in fields:
        s = norm(f)
        if not s:
            continue
        s = ADDR_PREFIX_RE.sub("", s)
        for _ in range(3):
            s2 = HOUSENUM_RE.sub("", DIRWORD_RE.sub("", s))
            if s2 == s:
                break
            s = s2
        s = s.split(",")[0].strip().rstrip(".").strip()
        s = re.sub(r"\s+(bt|bet|between)\s+.*$", "", s, flags=re.I)
        s = re.sub(r"\s+", " ", s).strip()
        s = re.sub(r"\b(av|ave)\b\.?$", "ave", s, flags=re.I)
        if s and s.lower() not in PLACEHOLDER_ADDR and len(s) > 1:
            return s
    return None

## 5. Relationships

Co-residents by normalised address, business links by employer text with a parenthetical
fallback, coworkers by employer, and groups by surname. All matching happens **within one
edition** -- an 1892 listing and a 1900 listing are never treated as related, even with an
identical name, because that is a different question.

In [5]:
# --------------------------------------------------------- relationships
def compute_relationships(df):
    entries = df.to_dict("records")
    by_edition = defaultdict(list)
    for e in entries:
        by_edition[e["edition"]].append(e)

    for group in by_edition.values():
        addr_g, emp_g, sur_g = defaultdict(list), defaultdict(list), defaultdict(list)
        biz_by_name = {}
        for e in group:
            a = normalize_address(e.get("residence_raw"), e.get("boarding_raw"), e.get("rooms_raw"))
            e["_addr"] = a
            if a:
                addr_g[a].append(e["id"])
            if norm(e.get("entry_type")) in ("business", "institution"):
                k = normalize_business(e.get("business_name") or e.get("raw_entry_text"))
                if k:
                    biz_by_name.setdefault(k, []).append(e["id"])
            aff = norm(e.get("employer")) or extract_paren_affiliation(e.get("raw_entry_text"))
            ek = normalize_business(aff) if aff else None
            e["_emp"] = ek
            if ek:
                emp_g[ek].append(e["id"])
            ln = nkey(e.get("last_name"))
            if ln:
                sur_g[ln].append(e["id"])

        for e in group:
            k = e["_emp"]
            linked = biz_by_name.get(k, []) if k else []
            if k and not linked:
                for bk, ids in biz_by_name.items():
                    if k in bk or bk in k:
                        linked = ids
                        break
            e["linked_business_ids"] = [i for i in linked if i != e["id"]]
            a = e["_addr"]
            e["coresident_ids"] = [i for i in addr_g.get(a, []) if i != e["id"]] if a else []
            e["coworker_ids"] = [i for i in emp_g.get(k, []) if i != e["id"]] if k else []
            ln = nkey(e.get("last_name"))
            e["same_surname_ids"] = [i for i in sur_g.get(ln, []) if i != e["id"]] if ln else []
            e.pop("_addr", None)
            e.pop("_emp", None)
    return entries

## 6. Data quality flags

Marks entries that probably need checking against the scan. Two independent signals, kept and
reported separately rather than merged into one number:

1. **Fragment** -- the transcription starts mid-word or mid-phrase, or a person entry has a
   surname and nothing else. These come from a box drawn across a line break, and the signal is
   close to deterministic: a mid-word start does not happen in a real printed listing.

2. **Duplicate** -- two neighbouring entries on the same page have near-identical full text.
   This one is genuinely harder, and it is worth walking through why, because an earlier version
   of it was wrong in a way that is easy to get wrong again.

   The first version required the two entries' names to match exactly before comparing text, on
   the theory that a true duplicate (one listing read twice by overlapping boxes) would parse to
   the same name both times. Tested against the full 5,621-entry set, that version found **zero**
   duplicates -- not because there weren't any, but because it was checking the wrong thing.
   Genuinely different people who happen to share a name (this directory has several: two
   different men named `Anderson John (c)`, a father-and-son `Andrus William C.` and
   `Andrus William C., jr.`) score up to 0.73 on text similarity, comfortably below any
   reasonable threshold. Meanwhile a real OCR duplicate -- the same box read twice -- more often
   disagrees on a letter *in the name itself* than agrees on it, e.g. `Boule Emery, carriage
   painter, h. 1418 Hardy.` next to `Boyle Emery, carriage painter, h. 1418 Hardy.` (0.98
   similarity, same address, same job -- almost certainly one listing, not two men named Emery
   at the same house). Requiring an exact name match filtered out exactly the cases it was meant
   to catch.

   So the current version drops the name-match requirement and instead thresholds on full-text
   similarity alone, at **0.92**. That number came from looking at every adjacent same-page pair
   in the real dataset: this is an alphabetized directory dense with siblings, parents and
   children, and common surnames at a shared address, so *genuinely different, genuinely similar*
   neighbours are common, not rare -- ratios up to about 0.85 are dominated by real, different
   people. 0.92 is roughly where that gives way to differences of only one or two characters, the
   signature of OCR noise on the same source text rather than two different people. There is no
   threshold that draws a clean line; 0.92 trades recall for precision on purpose, so the flag
   stays a credible prompt to look rather than noise to ignore.

On the full dataset this puts **113 entries (2.0%)** on the fragment signal and **46 entries
(0.8%)** on the duplicate signal, 159 total (2.8%) needing a look. Both are a prompt to check the
scan, not proof of an error -- similar text can and does legitimately recur in a directory.

In [6]:
# ------------------------------------------------------ data quality flags
def fragment_reason(e):
    """Signals that an entry is a piece of a listing rather than a whole one.
    These come straight from the observed failures: a box drawn across a line
    break yields text starting mid-word ("kermann Oscar"), starting mid-phrase
    ("attorney, notary, 9-11 Fox bldg"), or a surname with nothing after it
    ("Post, r. 1012 McKinney ave.")."""
    raw = norm(e.get("raw_entry_text"))
    t = norm(e.get("entry_type")) or "person"
    if not raw:
        return "empty transcription"
    if raw[0].islower():
        return "starts mid-word or mid-phrase"
    if not raw[0].isalpha():
        return "starts with punctuation or a number"
    if t == "person":
        if not nkey(e.get("last_name")):
            return "person entry with no surname"
        if not nkey(e.get("first_name")) and not norm(e.get("occupation_raw")) \
                and not norm(e.get("employer")):
            return "surname only, no given name or occupation"
    if t in ("business", "institution") and len(nkey(e.get("business_name"))) < 5:
        return "business name too short to be complete"
    return None


def flag_quality(entries, threshold=DUP_THRESHOLD):
    """Mark entries that probably need a human to check them against the scan.

    Two independent signals, kept and reported separately:
      - fragment: the entry itself looks like a partial listing (mid-word
        start, surname with nothing after it, etc). This one is close to
        deterministic -- a mid-word start is a box-boundary artifact, not
        something that occurs in a real printed listing.
      - duplicate: two neighbouring entries on the same page have near-
        identical full text. This one is a judgment call, not a fact: an
        earlier version of this check compared name fields for an exact
        match before comparing text, on the theory that a genuine duplicate
        (one listing read twice) would parse to the same name. Testing that
        against the real data showed the opposite -- it caught zero true
        duplicates and would have hidden real ones, because two OCR passes
        over the same box more often disagree on a letter in the name itself
        (e.g. "Boule Emery" / "Boyle Emery", same address, same job) than
        agree on it. So this checks raw text similarity only, at a threshold
        high enough (see DUP_THRESHOLD) to stay above the very real rate of
        genuinely similar, genuinely different neighbours in an alphabetized
        directory of families and common surnames -- it will still be wrong
        sometimes in both directions, which is why it is a flag to check,
        not a claim of an error.
    """
    for e in entries:
        e["fragment_flag"] = fragment_reason(e) or ""
        e["dup_flag"] = ""

    for i in range(1, len(entries)):
        a, b = entries[i - 1], entries[i]
        if a.get("source_page") != b.get("source_page"):
            continue
        ratio = difflib.SequenceMatcher(
            None, norm(a.get("raw_entry_text")), norm(b.get("raw_entry_text"))).ratio()
        if ratio >= threshold:
            note = f"near-identical to the neighbouring entry on this page, {ratio:.2f} text similarity"
            a["dup_flag"] = note
            b["dup_flag"] = note

    for e in entries:
        parts = [p for p in (e["fragment_flag"], e["dup_flag"]) if p]
        e["flag"] = "; ".join(parts)
    return entries

## 7. Derived fields for the charts

Precomputed in Python so the browser groups on the same tested parsing, and so the charts stay
fast at 5,000+ rows.

In [7]:
# ------------------------------------------------- derived chart fields
def add_derived(entries):
    """Precompute the keys the browser groups by, so the charts stay fast and
    reuse the same tested parsing as the Python side."""
    for e in entries:
        e["street"] = street_name(e.get("residence_raw"), e.get("boarding_raw"), e.get("rooms_raw")) or ""
        e["occ_key"] = norm(e.get("occupation_raw")).lower().rstrip(".,")
        e["emp_key"] = norm(e.get("employer")).rstrip(".,")
        if norm(e.get("residence_raw")):
            e["housing"] = "Own residence (r. / h.)"
        elif norm(e.get("boarding_raw")):
            e["housing"] = "Boarding (bds)"
        elif norm(e.get("rooms_raw")):
            e["housing"] = "Rooming (rms)"
        else:
            e["housing"] = "No address listed"
    return entries

## 8. The HTML template

Everything the dashboard does lives here: layout, styling, and the JavaScript for search,
sorting, the two Overview sections (the static-looking "At a glance" grid and the interactive
deep-dive chart), the checkbox filter dropdowns, and the CSV export. Deliberately
dependency-free -- no CDN, no chart library -- so the file keeps working offline and in ten
years' time.

In [8]:
HTML_TEMPLATE = r"""<!DOCTYPE html>
<html><head><meta charset="utf-8">
<title>City Directory -- Browse</title>
<style>
:root{--navy:#1B2A4A;--navy-dk:#121D33;--navy-lt:#41598A;--brass:#C9A227;--brass-lt:#E4C766;
      --card:#F3F5F9;--ink:#1E293B;--mute:#5B6B85;--border:#E3E7EF;--flag:#B4541F}
*{box-sizing:border-box}
body{font-family:Calibri,"Segoe UI",sans-serif;margin:0;background:#fff;color:var(--ink)}
header{background:var(--navy);color:#fff;padding:20px 28px 0}
header h1{margin:0 0 4px;font-family:Cambria,Georgia,serif;font-size:25px}
header p{margin:0;color:#AEB9CF;font-size:13px}
.stats{display:flex;gap:20px;margin-top:12px;flex-wrap:wrap}
.stat{background:rgba(255,255,255,.07);border-radius:8px;padding:7px 15px;min-width:104px}
.stat .num{font-family:Cambria,Georgia,serif;font-size:21px;font-weight:bold;color:var(--brass-lt)}
.stat .lbl{font-size:10.5px;color:#C9D1E3;margin-top:2px}
.tabs{display:flex;gap:6px;margin-top:14px}
.tab{background:transparent;border:none;color:#AEB9CF;font-size:14px;padding:9px 20px;cursor:pointer;
     border-radius:7px 7px 0 0;font-family:inherit}
.tab:hover{color:#fff}
.tab.active{background:#fff;color:var(--navy);font-weight:bold}
.toolbar{display:flex;gap:9px;padding:14px 28px;background:var(--card);
         border-bottom:1px solid var(--border);flex-wrap:wrap;align-items:center}
.toolbar input[type=text]{flex:1;min-width:210px;padding:9px 12px;border:1px solid #CBD2E0;
                          border-radius:6px;font-size:14px}
.toolbar select{padding:9px 10px;border:1px solid #CBD2E0;border-radius:6px;font-size:13.5px;background:#fff}
.toolbar label.chk{font-size:12.5px;color:var(--mute);display:flex;align-items:center;gap:5px;cursor:pointer}
.btn{background:var(--navy);color:#fff;border:none;border-radius:6px;padding:9px 14px;
     cursor:pointer;font-size:13px;font-family:inherit}
.btn:hover{background:var(--navy-dk)}
.btn.ghost{background:#fff;color:var(--navy);border:1px solid #CBD2E0}
.btn.ghost:hover{background:#EEF1F7}
.count{font-size:12.5px;color:var(--mute);margin-left:auto;white-space:nowrap}
.drill{display:none;padding:8px 28px;background:#FFF7E0;border-bottom:1px solid #EBDCA8;font-size:13px;color:#6B5518}
.drill b{color:#4A3A0C}
.drill button{margin-left:10px;background:none;border:1px solid #D9C67E;color:#6B5518;border-radius:5px;
              padding:2px 9px;cursor:pointer;font-size:12px;font-family:inherit}
.layout{display:flex;height:calc(100vh - 246px)}
.list-pane{flex:1.35;overflow-y:auto;border-right:1px solid var(--border)}
table{width:100%;border-collapse:collapse;font-size:13px}
thead th{position:sticky;top:0;background:#fff;border-bottom:2px solid var(--navy);text-align:left;
         padding:8px 10px;font-size:11.5px;text-transform:uppercase;letter-spacing:.03em;
         color:var(--mute);cursor:pointer;user-select:none;white-space:nowrap}
thead th:hover{background:#F0F3F9;color:var(--navy)}
thead th .arrow{color:var(--brass);font-size:10px;margin-left:3px}
tbody td{padding:7px 10px;border-bottom:1px solid var(--border)}
tbody tr{cursor:pointer}
tbody tr:hover{background:#F7F8FC}
tbody tr.active{background:#EAF0FB}
.marker{font-family:Consolas,monospace;font-size:12px;color:#7A3418}
.flagdot{color:var(--flag);font-weight:bold;cursor:help}
.badge{display:inline-block;font-size:10px;font-weight:bold;padding:2px 7px;border-radius:10px;text-transform:uppercase}
.badge-person{background:#E3E7EF;color:var(--mute)}
.badge-business{background:var(--brass-lt);color:var(--navy-dk)}
.badge-institution{background:#D7C4F0;color:#4B2E77}
.badge-cross_reference{background:#F5D0C5;color:#7A3418}
.badge-unclear{background:#EDEDED;color:#777}
.pager{display:flex;gap:8px;align-items:center;justify-content:center;padding:10px;
       border-top:1px solid var(--border);font-size:13px}
.pager button{background:var(--navy);color:#fff;border:none;border-radius:5px;padding:6px 14px;
              cursor:pointer;font-size:13px}
.pager button:disabled{background:#B7BECF;cursor:default}
.detail-pane{flex:1;overflow-y:auto;padding:20px 24px;background:#fff}
.detail-pane .placeholder{color:var(--mute);font-size:14px;margin-top:40px;text-align:center}
.detail-name{font-family:Cambria,Georgia,serif;font-size:22px;color:var(--navy);margin:0 0 2px}
.detail-raw{font-style:italic;color:var(--mute);font-size:12.5px;background:var(--card);
            padding:8px 10px;border-left:3px solid var(--brass);margin:10px 0 14px}
.warn{background:#FFF4EC;border-left:3px solid var(--flag);padding:8px 10px;font-size:12.5px;
      color:#7A3418;margin:0 0 14px}
.field-grid{display:grid;grid-template-columns:122px 1fr;gap:5px 10px;font-size:13.5px;margin-bottom:16px}
.field-grid b{color:var(--mute);font-weight:600}
abbr{text-decoration:underline dotted #A9B3C6;cursor:help}
.rel-section{margin-bottom:18px}
.rel-section h3{font-size:12.5px;text-transform:uppercase;letter-spacing:.03em;color:var(--navy);
                border-bottom:1px solid var(--border);padding-bottom:5px;margin-bottom:7px}
.rel-item{padding:6px 8px;border-radius:5px;font-size:13px;cursor:pointer;margin-bottom:3px}
.rel-item:hover{background:var(--card)}
.rel-item .rel-sub{color:var(--mute);font-size:11.5px}
.rel-empty{color:var(--mute);font-size:12.5px;font-style:italic}
.rel-more{font-size:11.5px;color:var(--mute);padding-left:8px}
.cite{font-size:11.5px;color:var(--mute);border-top:1px solid var(--border);padding-top:10px;margin-top:6px}
.cite code{background:var(--card);padding:2px 5px;border-radius:3px;font-size:11px}
#overview{display:none;padding:18px 28px 50px;overflow-y:auto;height:calc(100vh - 246px)}
.chart-controls{display:flex;gap:10px;flex-wrap:wrap;align-items:center;background:var(--card);
                padding:12px 14px;border-radius:8px;margin-bottom:18px}
.chart-controls label{font-size:12.5px;color:var(--mute);display:flex;align-items:center;gap:6px}
.chart-controls select,.chart-controls input[type=range]{font-size:13px}
.chart-wrap{max-width:1000px}
.chart-title{font-family:Cambria,Georgia,serif;font-size:19px;color:var(--navy);font-weight:bold;margin:0 0 3px}
.chart-sub{font-size:12.5px;color:var(--mute);margin:0 0 14px;line-height:1.5;max-width:840px}
.bar-row{cursor:pointer}
.bar-row:hover rect.bar{opacity:.78}
.chart-note{font-size:12px;color:var(--mute);margin-top:14px;line-height:1.55;max-width:840px;
            border-top:1px solid var(--border);padding-top:12px}
.mfilter{position:relative}
.mfilter-btn{white-space:nowrap}
.mf-count{background:var(--brass);color:var(--navy-dk);border-radius:9px;padding:0 6px;font-size:11px;margin:0 3px}
.mfilter-panel{display:none;position:absolute;top:calc(100% + 4px);left:0;background:#fff;border:1px solid var(--border);
  border-radius:8px;box-shadow:0 8px 24px rgba(20,32,60,.14);padding:10px 12px;z-index:40;min-width:220px;
  max-height:280px;overflow-y:auto}
.mfilter-panel.open{display:block}
.mfilter-panel label{display:flex;align-items:center;gap:7px;font-size:12.5px;padding:4px 2px;cursor:pointer;
  white-space:nowrap}
.mfilter-actions{display:flex;gap:10px;margin-top:6px;padding-top:6px;border-top:1px solid var(--border)}
.mfilter-actions button{background:none;border:none;color:var(--navy);font-size:11.5px;cursor:pointer;
  text-decoration:underline;padding:0;font-family:inherit}
.snapshot-head{display:flex;align-items:baseline;justify-content:space-between;margin:0 0 12px}
.snapshot-head h2{font-family:Cambria,Georgia,serif;font-size:19px;color:var(--navy);font-weight:bold;margin:0}
.snapshot-head p{font-size:12px;color:var(--mute);margin:0}
.snapshot-grid{display:grid;grid-template-columns:repeat(2,1fr);gap:16px;margin-bottom:30px}
.snapshot-card{background:var(--card);border-radius:10px;padding:14px 16px}
.snapshot-card h4{font-size:12.5px;text-transform:uppercase;letter-spacing:.03em;color:var(--navy);
  margin:0 0 10px}
.mini-row{display:flex;align-items:center;gap:8px;font-size:12px;margin-bottom:6px;cursor:pointer}
.mini-row:hover .mini-bar-fill{opacity:.8}
.mini-label{width:118px;flex-shrink:0;overflow:hidden;text-overflow:ellipsis;white-space:nowrap;color:var(--ink)}
.mini-track{flex:1;background:#E3E7EF;border-radius:4px;height:13px;overflow:hidden}
.mini-fill{background:var(--navy);height:100%;border-radius:4px}
.mini-count{width:36px;text-align:right;color:var(--mute);flex-shrink:0}
.divider{border:none;border-top:1px solid var(--border);margin:0 0 22px}
#glossary{display:none;position:fixed;right:0;top:0;bottom:0;width:370px;background:#fff;
          border-left:1px solid var(--border);box-shadow:-6px 0 22px rgba(20,32,60,.13);
          padding:20px 22px;overflow-y:auto;z-index:50}
#glossary h2{font-family:Cambria,Georgia,serif;color:var(--navy);font-size:19px;margin:0 0 4px}
#glossary p.lede{font-size:12.5px;color:var(--mute);margin:0 0 14px;line-height:1.5}
#glossary .g{display:grid;grid-template-columns:88px 1fr;gap:4px 10px;font-size:12.5px}
#glossary .g b{font-family:Consolas,monospace;color:var(--navy)}
#glossary .close{position:absolute;top:14px;right:16px;background:none;border:none;font-size:20px;
                 color:var(--mute);cursor:pointer}
#glossary h3{font-size:11.5px;text-transform:uppercase;letter-spacing:.03em;color:var(--mute);
             margin:18px 0 7px;border-bottom:1px solid var(--border);padding-bottom:4px}
footer{padding:10px 28px;font-size:11px;color:var(--mute);border-top:1px solid var(--border);line-height:1.5}
</style>
</head>
<body>

<header>
  <h1>City Directory -- Browse</h1>
  <p>Search by name, address, or occupation. Click a column heading to sort, or a bar in Overview to filter.</p>
  <div class="stats">
    <div class="stat"><div class="num">__N_ENTRIES__</div><div class="lbl">entries</div></div>
    <div class="stat"><div class="num">__N_PAGES__</div><div class="lbl">pages digitized</div></div>
    <div class="stat"><div class="num">__N_EDITIONS__</div><div class="lbl">edition(s)</div></div>
    <div class="stat"><div class="num">__N_PERSON__</div><div class="lbl">people</div></div>
    <div class="stat"><div class="num">__N_BIZ__</div><div class="lbl">businesses</div></div>
    <div class="stat"><div class="num">__N_LINKED__</div><div class="lbl">with a found connection</div></div>
    <div class="stat"><div class="num">__N_FLAGGED__</div><div class="lbl">needing review</div></div>
  </div>
  <div class="tabs">
    <button class="tab active" data-view="browse">Browse</button>
    <button class="tab" data-view="overview">Overview</button>
  </div>
</header>

<div id="browse">
  <div class="toolbar">
    <input type="text" id="searchBox" placeholder="Search name, address, occupation, employer...">
    <label class="chk"><input type="checkbox" id="fuzzyBox"> sounds-like</label>
    <div class="mfilter" id="typeMFilter">
      <button class="btn ghost mfilter-btn" id="typeMBtn" type="button">Type<span class="mf-count" id="typeMCount" style="display:none"></span> &#9662;</button>
      <div class="mfilter-panel" id="typeMPanel"></div>
    </div>
    <div class="mfilter" id="raceMFilter">
      <button class="btn ghost mfilter-btn" id="raceMBtn" type="button">Marker<span class="mf-count" id="raceMCount" style="display:none"></span> &#9662;</button>
      <div class="mfilter-panel" id="raceMPanel"></div>
    </div>
    <select id="flagFilter">
      <option value="">All rows</option>
      <option value="__CLEAN__">Hide rows needing review</option>
      <option value="__FLAG__">Only rows needing review</option>
    </select>
    <select id="editionFilter"><option value="">All editions</option></select>
    <select id="pageFilter"><option value="">All pages</option></select>
    <button class="btn ghost" id="glossaryBtn">Abbreviations</button>
    <button class="btn" id="exportBtn">Download CSV</button>
    <div class="count" id="resultCount"></div>
  </div>

  <div class="drill" id="drillBar"></div>

  <div class="layout">
    <div class="list-pane">
      <table><thead><tr id="headRow"></tr></thead><tbody id="rows"></tbody></table>
      <div class="pager">
        <button id="prevBtn">&larr; Prev</button><span id="pageLabel"></span>
        <button id="nextBtn">Next &rarr;</button>
      </div>
    </div>
    <div class="detail-pane" id="detailPane">
      <div class="placeholder">Click any entry on the left to see its full record and related entries.</div>
    </div>
  </div>
</div>

<div id="overview">
  <div class="snapshot-head">
    <h2>At a glance</h2>
    <p>Whole dataset, always -- click any bar to jump into Browse filtered to it.</p>
  </div>
  <div class="snapshot-grid" id="snapshotGrid"></div>
  <hr class="divider">
  <div class="snapshot-head">
    <h2>Explore a metric</h2>
    <p>Same charts, but you choose what's counted, how it's ordered, and how deep to go.</p>
  </div>
  <div class="chart-controls">
    <label>Show
      <select id="metricSel">
        <option value="occupation">Occupations</option>
        <option value="employer">Employers</option>
        <option value="street">Streets</option>
        <option value="surname">Surnames</option>
        <option value="marker">Markers as printed</option>
        <option value="housing">Housing type</option>
        <option value="type">Entry type</option>
        <option value="page">Entries per page</option>
      </select>
    </label>
    <label>Order
      <select id="orderSel">
        <option value="count">Most common first</option>
        <option value="share">Highest marker share first</option>
        <option value="alpha">A to Z</option>
      </select>
    </label>
    <label>Show top <b id="topNLabel">15</b>
      <input type="range" id="topN" min="5" max="40" value="15">
    </label>
    <label class="chk"><input type="checkbox" id="splitMarker"> split by marker</label>
    <label class="chk"><input type="checkbox" id="respectFilters" checked> follow Browse filters</label>
    <button class="btn ghost" id="chartCsvBtn">Download chart data</button>
  </div>
  <div class="chart-wrap">
    <h2 class="chart-title" id="chartTitle"></h2>
    <p class="chart-sub" id="chartSub"></p>
    <div id="chartHost"></div>
    <p class="chart-note" id="chartNote"></p>
  </div>
</div>

<div id="glossary">
  <button class="close" id="glossaryClose">&times;</button>
  <h2>Reading a directory entry</h2>
  <p class="lede">The 1900-1901 directory abbreviates almost everything. These are kept exactly as
  printed rather than expanded, so the digitized record matches the page. Hover any underlined
  abbreviation in an entry to see its meaning.</p>
  <div id="glossaryBody"></div>
</div>

<footer>
Digitized by an automated pipeline (OpenCV layout detection, Google Gemini transcription and field parsing). Relationships and the review flags are computed heuristically and are indicative, not authoritative.<br>
The Marker column reproduces the racial or ethnic identifier exactly as printed in the original directory, and is blank wherever the directory printed none. Nothing is inferred, standardised, or assigned. These are the 1900 publisher's terms, not descriptions of how any person identified themselves.
</footer>

<script>
const DATA = __DATA_JSON__;
const GLOSSARY = __GLOSSARY_JSON__;
const byId = {};
DATA.forEach(e => byId[e.id] = e);

/* ---------- helpers ---------- */
function displayName(e){
  if(e.Type === "person"){
    const ln = e["Last name"]||"", fn = e["First name"]||"";
    return (ln+" "+fn).trim() || "(unnamed)";
  }
  return e.Business || e.raw.slice(0,60);
}
function residenceOf(e){ return e.Residence || e.Boarding || e.Rooms || ""; }
function pageNum(e){ const m = String(e.page).match(/(\d+)\s*$/); return m ? parseInt(m[1]) : 0; }
function esc(s){ return String(s==null?"":s).replace(/[&<>"]/g, c => ({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;"}[c])); }

/* Soundex, so a name misread by OCR still matches what the user typed.
   "Achilles" and "Hemmes" will not match, but "Abell"/"Kbell" style single
   letter slips and spelling variants largely will. */
function soundex(s){
  s = String(s||"").toUpperCase().replace(/[^A-Z]/g,"");
  if(!s) return "";
  const code = c => "BFPV".includes(c)?"1":"CGJKQSXZ".includes(c)?"2":"DT".includes(c)?"3":
                    c==="L"?"4":"MN".includes(c)?"5":c==="R"?"6":"";
  let out = s[0], prev = code(s[0]);
  for(let i=1;i<s.length;i++){
    const c = code(s[i]);
    if(c && c!==prev) out += c;
    if(!"HW".includes(s[i])) prev = c;
    if(out.length===4) break;
  }
  return (out+"000").slice(0,4);
}
const SOUNDEX_CACHE = new Map();
function nameCodes(e){
  if(SOUNDEX_CACHE.has(e.id)) return SOUNDEX_CACHE.get(e.id);
  const codes = [e["Last name"], e["First name"], e.Business]
    .filter(Boolean).map(x => String(x).split(/\s+/).map(soundex)).flat();
  SOUNDEX_CACHE.set(e.id, codes);
  return codes;
}

/* ---------- state ---------- */
let filtered = DATA.slice();
let currentPageNum = 0, activeId = null;
let sortKey = null, sortDir = 1;
let drill = null;                 /* {field, value, label} set by clicking a bar */
const PAGE_SIZE = 50;

const COLUMNS = [
  {key:"name", label:"Name / Business", get:e=>displayName(e),
   html:e=>(e.flag?`<span class="flagdot" title="${esc(e.flag)}">&#9873;</span> `:"")+esc(displayName(e))},
  {key:"type", label:"Type", get:e=>e.Type||"",
   html:e=>`<span class="badge badge-${e.Type||"unclear"}">${esc(e.Type||"")}</span>`},
  {key:"race", label:"Marker", get:e=>e.Race||"",
   html:e=>e.Race?`<span class="marker">${esc(e.Race)}</span>`:""},
  {key:"occupation", label:"Occupation", get:e=>e.Occupation||"", html:e=>esc(e.Occupation||"")},
  {key:"employer", label:"Employer", get:e=>e.Employer||"", html:e=>esc(e.Employer||"")},
  {key:"residence", label:"Residence", get:e=>residenceOf(e), html:e=>esc(residenceOf(e))},
  {key:"page", label:"Page", get:e=>pageNum(e), html:e=>esc(e.page)},
];

/* ---------- tabs ---------- */
document.querySelectorAll(".tab").forEach(btn=>{
  btn.addEventListener("click",()=>{
    document.querySelectorAll(".tab").forEach(b=>b.classList.remove("active"));
    btn.classList.add("active");
    const v = btn.dataset.view;
    document.getElementById("browse").style.display   = v==="browse"?"block":"none";
    document.getElementById("overview").style.display = v==="overview"?"block":"none";
    if(v==="overview"){ renderSnapshot(); renderChart(); }
  });
});

/* ---------- sortable header ---------- */
const headRow = document.getElementById("headRow");
function renderHeader(){
  headRow.innerHTML = COLUMNS.map(c=>{
    const arrow = sortKey===c.key ? `<span class="arrow">${sortDir===1?"▲":"▼"}</span>` : "";
    return `<th data-key="${c.key}" title="Click to sort by ${c.label}">${c.label}${arrow}</th>`;
  }).join("");
  headRow.querySelectorAll("th").forEach(th=>{
    th.addEventListener("click",()=>{
      const k = th.dataset.key;
      if(sortKey===k) sortDir=-sortDir; else {sortKey=k; sortDir=1;}
      applySort(); currentPageNum=0; renderHeader(); renderRows();
    });
  });
}
function applySort(){
  if(!sortKey) return;
  const col = COLUMNS.find(c=>c.key===sortKey);
  filtered.sort((a,b)=>{
    const va=col.get(a), vb=col.get(b);
    /* blanks always last, in either direction */
    const ea=(va===""||va==null), eb=(vb===""||vb==null);
    if(ea&&eb) return 0; if(ea) return 1; if(eb) return -1;
    if(typeof va==="number"&&typeof vb==="number") return (va-vb)*sortDir;
    return String(va).localeCompare(String(vb),undefined,{sensitivity:"base"})*sortDir;
  });
}

/* ---------- filters ---------- */
const editionFilterEl = document.getElementById("editionFilter");
[...new Set(DATA.map(e=>e.edition))].forEach(ed=>{
  const o=document.createElement("option"); o.value=ed; o.textContent=ed; editionFilterEl.appendChild(o);
});
const pageFilterEl = document.getElementById("pageFilter");
function refreshPageOptions(){
  const ed = editionFilterEl.value;
  const pages=[...new Set(DATA.filter(e=>!ed||e.edition===ed).map(e=>e.page))];
  pageFilterEl.innerHTML='<option value="">All pages</option>'+
    pages.map(p=>`<option value="${esc(p)}">${esc(p)}</option>`).join("");
}
refreshPageOptions();

/* ---------- multi-select checkbox dropdowns (Type, Marker) ---------- */
const selectedTypes = new Set();
const selectedMarkers = new Set();
const NONE_MARKER = "__NONE__";

function buildMFilter(panelId, btnId, countId, values, selectedSet, labelFn){
  const panel = document.getElementById(panelId);
  panel.innerHTML = values.map(v=>
    `<label><input type="checkbox" value="${esc(v)}"> ${esc(labelFn(v))}</label>`).join("") +
    `<div class="mfilter-actions"><button type="button" data-act="all">All</button><button type="button" data-act="none">Clear</button></div>`;
  function syncCount(){
    const c = document.getElementById(countId);
    if(selectedSet.size && selectedSet.size < values.length){ c.style.display="inline"; c.textContent=selectedSet.size; }
    else c.style.display="none";
  }
  panel.querySelectorAll('input[type=checkbox]').forEach(cb=>{
    cb.addEventListener("change",()=>{
      if(cb.checked) selectedSet.add(cb.value); else selectedSet.delete(cb.value);
      syncCount(); applyFilters();
    });
  });
  panel.querySelector('[data-act="all"]').addEventListener("click",()=>{
    selectedSet.clear();
    panel.querySelectorAll('input[type=checkbox]').forEach(cb=>{ cb.checked=true; selectedSet.add(cb.value); });
    syncCount(); applyFilters();
  });
  panel.querySelector('[data-act="none"]').addEventListener("click",()=>{
    selectedSet.clear();
    panel.querySelectorAll('input[type=checkbox]').forEach(cb=>cb.checked=false);
    syncCount(); applyFilters();
  });
  document.getElementById(btnId).addEventListener("click",(ev)=>{
    ev.stopPropagation();
    const willOpen = !panel.classList.contains("open");
    document.querySelectorAll(".mfilter-panel.open").forEach(p=>p.classList.remove("open"));
    if(willOpen) panel.classList.add("open");
  });
}
document.addEventListener("click",()=>document.querySelectorAll(".mfilter-panel.open").forEach(p=>p.classList.remove("open")));
document.querySelectorAll(".mfilter-panel").forEach(p=>p.addEventListener("click",ev=>ev.stopPropagation()));

const TYPE_VALUES = [...new Set(DATA.map(e=>e.Type||"person"))].sort();
buildMFilter("typeMPanel","typeMBtn","typeMCount",TYPE_VALUES,selectedTypes, v=>v);

const MARKER_VALUES = [NONE_MARKER, ...[...new Set(DATA.map(e=>e.Race).filter(Boolean))].sort()];
buildMFilter("raceMPanel","raceMBtn","raceMCount",MARKER_VALUES,selectedMarkers,
  v=>v===NONE_MARKER?"(none printed)":v);

function matchesSearch(e,q,fuzzy){
  if(!q) return true;
  const hay=[displayName(e),e.Occupation,e.Employer,e.Residence,e.Boarding,e.Rooms,e.raw]
    .filter(Boolean).join(" ").toLowerCase();
  if(hay.includes(q)) return true;
  if(fuzzy){
    const qc = q.split(/\s+/).filter(Boolean).map(soundex).filter(Boolean);
    if(!qc.length) return false;
    const ec = nameCodes(e);
    return qc.some(c=>ec.includes(c));
  }
  return false;
}

function applyFilters(){
  const q=document.getElementById("searchBox").value.trim().toLowerCase();
  const fuzzy=document.getElementById("fuzzyBox").checked;
  const flagF=document.getElementById("flagFilter").value;
  const edition=editionFilterEl.value, page=pageFilterEl.value;
  filtered = DATA.filter(e =>
    matchesSearch(e,q,fuzzy) &&
    (!selectedTypes.size || selectedTypes.has(e.Type||"person")) &&
    (!selectedMarkers.size || selectedMarkers.has(e.Race||NONE_MARKER)) &&
    (!flagF || (flagF==="__FLAG__"? !!e.flag : !e.flag)) &&
    (!edition||e.edition===edition) &&
    (!page||e.page===page) &&
    (!drill || String(drillValue(e,drill.field))===String(drill.value))
  );
  applySort(); currentPageNum=0; renderRows(); renderDrillBar();
  if(document.getElementById("overview").style.display==="block") renderChart();
}

/* ---------- drill-down from a chart bar ---------- */
function drillValue(e,field){
  switch(field){
    case "occupation": return e.occ_key||"";
    case "employer":   return e.emp_key||"";
    case "street":     return e.street||"";
    case "surname":    return (e["Last name"]||"").toLowerCase();
    case "marker":     return e.Race||"(none printed)";
    case "housing":    return e.housing||"";
    case "type":       return e.Type||"";
    case "page":       return e.page||"";
  }
  return "";
}
function renderDrillBar(){
  const bar=document.getElementById("drillBar");
  if(!drill){ bar.style.display="none"; return; }
  bar.style.display="block";
  bar.innerHTML=`Filtered from the Overview chart: <b>${esc(drill.label)}</b>
                 <button id="clearDrill">clear</button>`;
  document.getElementById("clearDrill").onclick=()=>{ drill=null; applyFilters(); };
}

/* ---------- table ---------- */
function renderRows(){
  const tbody=document.getElementById("rows");
  const start=currentPageNum*PAGE_SIZE;
  tbody.innerHTML=filtered.slice(start,start+PAGE_SIZE).map(e=>
    `<tr data-id="${e.id}" class="${e.id===activeId?"active":""}">`+
    COLUMNS.map(c=>`<td>${c.html(e)}</td>`).join("")+`</tr>`).join("");
  tbody.querySelectorAll("tr").forEach(tr=>
    tr.addEventListener("click",()=>showDetail(parseInt(tr.dataset.id))));
  const totalPages=Math.max(1,Math.ceil(filtered.length/PAGE_SIZE));
  document.getElementById("pageLabel").textContent=`Page ${currentPageNum+1} of ${totalPages}`;
  document.getElementById("prevBtn").disabled=currentPageNum===0;
  document.getElementById("nextBtn").disabled=currentPageNum>=totalPages-1;
  document.getElementById("resultCount").textContent=`${filtered.length.toLocaleString()} matching entries`;
}

/* ---------- detail ---------- */
function annotate(text){
  /* underline abbreviations the glossary knows, so an outside reader can
     hover "bds" or "(c)" instead of guessing */
  let out = esc(text);
  Object.keys(GLOSSARY.terms)
    .sort((a,b)=>b.length-a.length)
    .forEach(term=>{
      const safe = term.replace(/[.*+?^${}()|[\]\\]/g,"\\$&");
      const re = new RegExp("(^|[\\s(,;])("+safe+")(?=[\\s.,;)]|$)","gi");
      out = out.replace(re,(m,pre,t)=>`${pre}<abbr title="${esc(GLOSSARY.terms[term])}">${t}</abbr>`);
    });
  return out;
}
function relList(ids,emptyMsg,cap=8){
  if(!ids||!ids.length) return `<div class="rel-empty">${emptyMsg}</div>`;
  const shown=ids.slice(0,cap).map(id=>{
    const e=byId[id]; if(!e) return "";
    return `<div class="rel-item" data-id="${id}">${esc(displayName(e))}
      <div class="rel-sub">${esc(e.Occupation||e.Type)} -- ${esc(e.page)}</div></div>`;
  }).join("");
  return shown+(ids.length>cap?`<div class="rel-more">+ ${ids.length-cap} more</div>`:"");
}
function showDetail(id){
  activeId=id;
  const e=byId[id];
  if(history.replaceState) history.replaceState(null,"","#entry-"+id);
  const pane=document.getElementById("detailPane");
  pane.innerHTML=`
    <h2 class="detail-name">${esc(displayName(e))}</h2>
    <span class="badge badge-${e.Type||"unclear"}">${esc(e.Type||"")}</span>
    <div class="detail-raw">${annotate(e.raw)}</div>
    ${e.flag?`<div class="warn">
      ${e.fragment?`<div><b>Possible fragment:</b> ${esc(e.fragment)}</div>`:""}
      ${e.dup?`<div><b>Possible duplicate:</b> ${esc(e.dup)}</div>`:""}
      A heuristic flag -- worth a quick check against the scanned page, not proof of an error.</div>`:""}
    <div class="field-grid">
      ${e.Occupation?`<b>Occupation</b><span>${annotate(e.Occupation)}</span>`:""}
      ${e.Employer?`<b>Employer</b><span>${annotate(e.Employer)}</span>`:""}
      ${e.Workplace?`<b>Workplace</b><span>${annotate(e.Workplace)}</span>`:""}
      ${e.Residence?`<b>Residence</b><span>${annotate(e.Residence)}</span>`:""}
      ${e.Boarding?`<b>Boarding</b><span>${annotate(e.Boarding)}</span>`:""}
      ${e.Rooms?`<b>Rooms</b><span>${annotate(e.Rooms)}</span>`:""}
      ${e.Race?`<b>Marker as printed</b><span class="marker">${esc(e.Race)}</span>`:""}
      ${e.Ownership?`<b>Ownership</b><span>${esc(e.Ownership)}</span>`:""}
      ${e.Notes?`<b>Notes</b><span>${annotate(e.Notes)}</span>`:""}
      <b>Edition</b><span>${esc(e.edition)}</span>
      <b>Source page</b><span>${esc(e.page)}</span>
    </div>
    <div class="rel-section"><h3>Works for / affiliated business</h3>
      ${relList(e.linked_business_ids,"No matching business entry found.")}</div>
    <div class="rel-section"><h3>Lives at the same address</h3>
      ${relList(e.coresident_ids,"No other entries share this address.")}</div>
    <div class="rel-section"><h3>Same employer (coworkers)</h3>
      ${relList(e.coworker_ids,"No coworker matches found.")}</div>
    <div class="rel-section"><h3>Same surname</h3>
      ${relList(e.same_surname_ids,"No same-surname matches.",12)}</div>
    <div class="cite">Cite as: <code>${esc(displayName(e))}, ${esc(e.edition)} Houston city
      directory, ${esc(e.page)}, entry ${e.id}.</code></div>`;
  pane.querySelectorAll(".rel-item").forEach(item=>
    item.addEventListener("click",()=>showDetail(parseInt(item.dataset.id))));
  renderRows();
}

/* ---------- CSV export ---------- */
function toCsv(rows,cols){
  const q = v => `"${String(v==null?"":v).replace(/"/g,'""')}"`;
  return [cols.map(q).join(",")].concat(
    rows.map(r=>cols.map(c=>q(r[c])).join(","))).join("\n");
}
function download(name,text){
  const blob=new Blob([text],{type:"text/csv;charset=utf-8;"});
  const a=document.createElement("a");
  a.href=URL.createObjectURL(blob); a.download=name; a.click(); URL.revokeObjectURL(a.href);
}
document.getElementById("exportBtn").addEventListener("click",()=>{
  const cols=["id","edition","page","Type","Last name","First name","Business","Race","Occupation",
              "Employer","Workplace","Residence","Boarding","Rooms","Ownership","Notes",
              "flag","fragment","dup","raw"];
  download(`directory_selection_${filtered.length}_rows.csv`, toCsv(filtered,cols));
});

/* ---------- glossary ---------- */
(function buildGlossary(){
  const host=document.getElementById("glossaryBody");
  host.innerHTML = GLOSSARY.groups.map(g =>
    `<h3>${esc(g.name)}</h3><div class="g">` +
    g.items.map(k=>`<b>${esc(k)}</b><span>${esc(GLOSSARY.terms[k])}</span>`).join("") +
    `</div>`).join("");
})();
document.getElementById("glossaryBtn").onclick=()=>document.getElementById("glossary").style.display="block";
document.getElementById("glossaryClose").onclick=()=>document.getElementById("glossary").style.display="none";

/* ---------- interactive chart ---------- */
const METRICS = {
  occupation:{field:"occupation", title:"Occupations",
    sub:"Occupations as abbreviated in the original (lab, clk, servt, bkpr). Click any bar to see those people.",
    axis:"People listed with this occupation"},
  employer:{field:"employer", title:"Employers",
    sub:"Counted from the employer field. The same firm can appear under more than one abbreviation.",
    axis:"People listed with this employer"},
  street:{field:"street", title:"Streets",
    sub:"Street parsed from the address. Corner and cross-street forms are reduced to the first street named.",
    axis:"Entries with an address on this street"},
  surname:{field:"surname", title:"Surnames",
    sub:"Most common surnames. Useful as a starting point for family research, though OCR errors split some names.",
    axis:"Entries with this surname"},
  marker:{field:"marker", title:"Markers as printed",
    sub:"The racial or ethnic identifiers exactly as the directory printed them, deliberately not merged into one category.",
    axis:"Entries"},
  housing:{field:"housing", title:"Housing type",
    sub:"The directory distinguishes a residence (r. / h.) from boarding (bds) and rooming (rms).",
    axis:"Entries"},
  type:{field:"type", title:"Entry type",
    sub:"How each entry was classified during parsing.", axis:"Entries"},
  page:{field:"page", title:"Entries per page",
    sub:"Per-page yield. Pages at or near zero are full-page advertisements or dividers.",
    axis:"Entries recovered"},
};

function aggregate(rows,field){
  const map=new Map();
  rows.forEach(e=>{
    const v=drillValue(e,field);
    if(v==="") return;
    if(!map.has(v)) map.set(v,{label:v,total:0,marked:0});
    const o=map.get(v); o.total++; if(e.Race) o.marked++;
  });
  return [...map.values()];
}

/* ---------- "At a glance" snapshot: a few charts visible at once, always
   over the full dataset, no controls -- for the reader who wants the shape
   of the data in one look before diving into the single deep-dive chart. */
const SNAPSHOT_METRICS = [
  {field:"type", title:"Entry type", cap:8},
  {field:"housing", title:"Housing type", cap:8},
  {field:"occupation", title:"Top occupations", cap:8},
  {field:"street", title:"Top streets", cap:8},
];
function renderSnapshot(){
  const host = document.getElementById("snapshotGrid");
  host.innerHTML = SNAPSHOT_METRICS.map(m=>{
    const data = aggregate(DATA,m.field).sort((a,b)=>b.total-a.total).slice(0,m.cap);
    const max = Math.max(1,...data.map(d=>d.total));
    const rows = data.map(d=>{
      const label = String(d.label).length>22?String(d.label).slice(0,21)+"…":String(d.label);
      const pct = Math.round(d.total/max*100);
      return `<div class="mini-row" data-field="${esc(m.field)}" data-value="${esc(d.label)}"
        title="${esc(d.label)} -- ${d.total} entries. Click to filter.">
        <span class="mini-label">${esc(label)}</span>
        <span class="mini-track"><span class="mini-fill" style="width:${pct}%"></span></span>
        <span class="mini-count">${d.total}</span></div>`;
    }).join("") || `<div class="rel-empty">No data.</div>`;
    return `<div class="snapshot-card"><h4>${esc(m.title)}</h4>${rows}</div>`;
  }).join("");
  host.querySelectorAll(".mini-row").forEach(row=>{
    row.addEventListener("click",()=>{
      const field = row.dataset.field, value = row.dataset.value;
      const title = SNAPSHOT_METRICS.find(m=>m.field===field).title.replace(/^Top /,"").replace(/s$/,"");
      drill = {field, value, label:`${title}: ${value}`};
      document.getElementById("metricSel").value = field;
      document.querySelector('.tab[data-view="browse"]').click();
      applyFilters();
    });
  });
}

function renderChart(){
  const metricKey=document.getElementById("metricSel").value;
  const spec=METRICS[metricKey];
  const order=document.getElementById("orderSel").value;
  const topN=parseInt(document.getElementById("topN").value);
  const split=document.getElementById("splitMarker").checked;
  const respect=document.getElementById("respectFilters").checked;
  document.getElementById("topNLabel").textContent=topN;

  const source=respect?filtered:DATA;
  let data=aggregate(source,spec.field);
  const grandTotal=data.reduce((s,d)=>s+d.total,0);

  if(order==="count") data.sort((a,b)=>b.total-a.total);
  else if(order==="alpha") data.sort((a,b)=>String(a.label).localeCompare(String(b.label)));
  else data.sort((a,b)=>(b.marked/b.total)-(a.marked/a.total) || b.total-a.total);
  if(order==="share") data=data.filter(d=>d.total>=5);
  data=data.slice(0,topN);

  document.getElementById("chartTitle").textContent=spec.title;
  document.getElementById("chartSub").textContent=spec.sub;
  document.getElementById("chartNote").innerHTML =
    `Showing ${data.length} of ${aggregate(source,spec.field).length} distinct values, `+
    `from ${source.length.toLocaleString()} entries`+
    (respect?" matching the current Browse filters":" (all entries)")+". "+
    (split?`Gold is the share of entries the directory printed a racial marker against; navy is the rest. `:"")+
    `These describe what the directory recorded, filtered through an automated pipeline, so they carry `+
    `both the publisher's conventions and some transcription error.`;

  const host=document.getElementById("chartHost");
  if(!data.length){ host.innerHTML=`<p class="rel-empty">Nothing to chart for this selection.</p>`; return; }

  const rowH=26, padL=190, padR=70, padT=8, padB=34, W=940;
  const H=padT+padB+data.length*rowH;
  const max=Math.max(...data.map(d=>d.total));
  const bw=W-padL-padR;
  const ticks=4;

  let svg=`<svg viewBox="0 0 ${W} ${H}" width="100%" height="${H}" font-family="Calibri,sans-serif">`;
  for(let i=0;i<=ticks;i++){
    const x=padL+bw*i/ticks;
    svg+=`<line x1="${x}" y1="${padT}" x2="${x}" y2="${H-padB}" stroke="#DDE2EC"/>`;
    svg+=`<text x="${x}" y="${H-padB+16}" font-size="11" fill="#5B6B85" text-anchor="middle">${Math.round(max*i/ticks)}</text>`;
  }
  svg+=`<text x="${padL+bw/2}" y="${H-6}" font-size="11.5" fill="#5B6B85" text-anchor="middle">${esc(spec.axis)}</text>`;

  data.forEach((d,i)=>{
    const y=padT+i*rowH, h=rowH-8;
    const wTot=Math.max(1,bw*d.total/max);
    const wMark=bw*d.marked/max;
    const pct=d.total?Math.round(d.marked/d.total*100):0;
    const label=String(d.label).length>28?String(d.label).slice(0,27)+"…":String(d.label);
    svg+=`<g class="bar-row" data-value="${esc(d.label)}">`;
    svg+=`<title>${esc(d.label)} -- ${d.total} entries${split||order==="share"?`, ${pct}% with a printed marker`:""}. Click to filter.</title>`;
    svg+=`<text x="${padL-9}" y="${y+h/2+4}" font-size="12.5" fill="#1E293B" text-anchor="end">${esc(label)}</text>`;
    if(split){
      svg+=`<rect class="bar" x="${padL}" y="${y}" width="${wTot}" height="${h}" fill="#41598A"/>`;
      if(wMark>0) svg+=`<rect class="bar" x="${padL}" y="${y}" width="${wMark}" height="${h}" fill="#C9A227"/>`;
    } else {
      svg+=`<rect class="bar" x="${padL}" y="${y}" width="${wTot}" height="${h}" fill="#1B2A4A"/>`;
    }
    svg+=`<text x="${padL+wTot+7}" y="${y+h/2+4}" font-size="11.5" fill="#5B6B85">${d.total}${split?`  (${pct}%)`:""}</text>`;
    svg+=`</g>`;
  });
  svg+=`</svg>`;
  host.innerHTML=svg;

  host.querySelectorAll(".bar-row").forEach(g=>{
    g.addEventListener("click",()=>{
      drill={field:spec.field,value:g.dataset.value,label:`${spec.title.replace(/s$/,"")}: ${g.dataset.value}`};
      document.querySelector('.tab[data-view="browse"]').click();
      applyFilters();
    });
  });
}

document.getElementById("chartCsvBtn").addEventListener("click",()=>{
  const spec=METRICS[document.getElementById("metricSel").value];
  const respect=document.getElementById("respectFilters").checked;
  const rows=aggregate(respect?filtered:DATA,spec.field)
    .sort((a,b)=>b.total-a.total)
    .map(d=>({value:d.label,entries:d.total,with_printed_marker:d.marked}));
  download(`chart_${spec.field}.csv`, toCsv(rows,["value","entries","with_printed_marker"]));
});

["metricSel","orderSel","splitMarker","respectFilters"].forEach(id=>
  document.getElementById(id).addEventListener("change",renderChart));
document.getElementById("topN").addEventListener("input",renderChart);

/* ---------- wiring ---------- */
["searchBox"].forEach(id=>document.getElementById(id).addEventListener("input",applyFilters));
["fuzzyBox","flagFilter","pageFilter"].forEach(id=>
  document.getElementById(id).addEventListener("change",applyFilters));
editionFilterEl.addEventListener("change",()=>{refreshPageOptions();applyFilters();});
document.getElementById("prevBtn").addEventListener("click",()=>{if(currentPageNum>0){currentPageNum--;renderRows();}});
document.getElementById("nextBtn").addEventListener("click",()=>{
  if(currentPageNum<Math.ceil(filtered.length/PAGE_SIZE)-1){currentPageNum++;renderRows();}});

renderHeader();
renderRows();
if(location.hash.startsWith("#entry-")){
  const id=parseInt(location.hash.slice(7));
  if(byId[id]) showDetail(id);
}
</script>
</body></html>
"""

## 9. Building the HTML

In [9]:
FIELD_MAP = {
    "Last name": "last_name", "First name": "first_name", "Business": "business_name",
    "Race": "racial_marker_raw", "Occupation": "occupation_raw", "Employer": "employer",
    "Workplace": "workplace_address", "Residence": "residence_raw", "Boarding": "boarding_raw",
    "Rooms": "rooms_raw", "Ownership": "ownership_type", "Type": "entry_type", "Notes": "notes",
}


def to_records(entries):
    out = []
    for e in entries:
        r = {"id": int(e["id"]), "page": norm(e.get("source_page")),
             "raw": norm(e.get("raw_entry_text")), "edition": norm(e.get("edition")),
             "flag": e.get("flag", ""), "fragment": e.get("fragment_flag", ""),
             "dup": e.get("dup_flag", ""), "street": e.get("street", ""),
             "occ_key": e.get("occ_key", ""), "emp_key": e.get("emp_key", ""),
             "housing": e.get("housing", "")}
        for label, field in FIELD_MAP.items():
            r[label] = norm(e.get(field))
        r["Type"] = r["Type"] or "person"
        for k in ("linked_business_ids", "coresident_ids", "coworker_ids", "same_surname_ids"):
            r[k] = [int(i) for i in e[k]]
        out.append(r)
    return out


def build_html(entries, output_path, template=HTML_TEMPLATE, glossary=GLOSSARY):
    recs = to_records(entries)
    html = template
    for token, value in [
        ("__N_ENTRIES__", f"{len(recs):,}"),
        ("__N_PAGES__", str(len({r['page'] for r in recs}))),
        ("__N_EDITIONS__", str(len({r['edition'] for r in recs}))),
        ("__N_PERSON__", f"{sum(1 for r in recs if r['Type'] == 'person'):,}"),
        ("__N_BIZ__", f"{sum(1 for r in recs if r['Type'] in ('business', 'institution')):,}"),
        ("__N_LINKED__", f"{sum(1 for r in recs if r['coresident_ids'] or r['coworker_ids'] or r['linked_business_ids']):,}"),
        ("__N_FLAGGED__", f"{sum(1 for r in recs if r['flag']):,}"),
        ("__GLOSSARY_JSON__", json.dumps(glossary, ensure_ascii=False)),
        ("__DATA_JSON__", json.dumps(recs, ensure_ascii=False)),
    ]:
        html = html.replace(token, value)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(html, encoding="utf-8")
    return output_path

## 10. Building the Excel workbook

In [10]:
# ----------------------------------------------------------------- Excel
def safe(v):
    """Excel reads a leading =, +, - or @ as a formula. One residence OCR'd as
    '= 1000 Gray ave.' and produced #VALUE!, so force those to stay text."""
    v = norm(v)
    return "'" + v if v[:1] in ("=", "+", "-", "@") else v


def build_xlsx(entries, output_path):
    by_id = {e["id"]: e for e in entries}

    def disp(e):
        if (norm(e.get("entry_type")) or "person") == "person":
            return (f"{norm(e.get('last_name'))} {norm(e.get('first_name'))}".strip()) or "(unnamed)"
        return norm(e.get("business_name")) or norm(e.get("raw_entry_text"))[:60]

    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = "Directory"
    headers = ["ID", "Edition", "Page", "Type", "Last Name", "First Name", "Business",
               "Marker as printed", "Occupation", "Employer", "Workplace", "Residence",
               "Boarding", "Rooms", "Street", "Housing", "Ownership", "Notes",
               "Needs review", "Fragment flag", "Duplicate flag", "Raw Text", "Linked Business",
               "Co-resident Count", "Coworker Count", "Same-Surname Count"]
    ws.append(headers)
    for c in range(1, len(headers) + 1):
        cell = ws.cell(row=1, column=c)
        cell.font = Font(name="Arial", bold=True, color="FFFFFF")
        cell.fill = PatternFill("solid", fgColor="1B2A4A")
        cell.alignment = Alignment(vertical="center")
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = f"A1:{get_column_letter(len(headers))}{len(entries) + 1}"

    for e in entries:
        linked = "; ".join(disp(by_id[i]) for i in e["linked_business_ids"][:3] if i in by_id)
        ws.append([
            e["id"], safe(e.get("edition")), safe(e.get("source_page")),
            safe(e.get("entry_type")) or "person", safe(e.get("last_name")),
            safe(e.get("first_name")), safe(e.get("business_name")),
            safe(e.get("racial_marker_raw")), safe(e.get("occupation_raw")),
            safe(e.get("employer")), safe(e.get("workplace_address")),
            safe(e.get("residence_raw")), safe(e.get("boarding_raw")), safe(e.get("rooms_raw")),
            safe(e.get("street")), safe(e.get("housing")), safe(e.get("ownership_type")),
            safe(e.get("notes")), safe(e.get("flag")), safe(e.get("fragment_flag")),
            safe(e.get("dup_flag")), safe(e.get("raw_entry_text")),
            safe(linked), len(e["coresident_ids"]), len(e["coworker_ids"]), len(e["same_surname_ids"]),
        ])

    widths = [6, 12, 10, 11, 14, 14, 20, 16, 16, 18, 16, 22, 16, 14, 14, 16, 11, 18, 34, 30, 30, 40, 22, 10, 10, 10]
    for i, w in enumerate(widths, start=1):
        ws.column_dimensions[get_column_letter(i)].width = w
    for row in ws.iter_rows(min_row=2, max_col=len(headers)):
        for cell in row:
            cell.font = Font(name="Arial", size=10)

    last = len(entries) + 1
    col = {name: get_column_letter(i + 1) for i, name in enumerate(headers)}
    ws2 = wb.create_sheet("Summary")
    ws2["A1"] = "City Directory -- Digitization Summary"
    ws2["A1"].font = Font(name="Arial", bold=True, size=14, color="1B2A4A")
    ws2.merge_cells("A1:C1")
    metrics = [
        ("Total entries", f"=COUNTA(Directory!A2:A{last})"),
        ("Editions", f"=SUMPRODUCT(1/COUNTIF(Directory!B2:B{last},Directory!B2:B{last}))"),
        ("Pages digitized", f"=SUMPRODUCT(1/COUNTIF(Directory!C2:C{last},Directory!C2:C{last}))"),
        ("Person entries", f'=COUNTIF(Directory!D2:D{last},"person")'),
        ("Business entries", f'=COUNTIF(Directory!D2:D{last},"business")'),
        ("Institution entries", f'=COUNTIF(Directory!D2:D{last},"institution")'),
        ("Entries with a printed marker", f'=COUNTIF(Directory!{col["Marker as printed"]}2:{col["Marker as printed"]}{last},"?*")'),
        ("Entries needing review", f'=COUNTIF(Directory!{col["Needs review"]}2:{col["Needs review"]}{last},"?*")'),
        ("  -- flagged as a fragment", f'=COUNTIF(Directory!{col["Fragment flag"]}2:{col["Fragment flag"]}{last},"?*")'),
        ("  -- flagged as a possible duplicate", f'=COUNTIF(Directory!{col["Duplicate flag"]}2:{col["Duplicate flag"]}{last},"?*")'),
        ("Entries with a co-resident", f'=COUNTIF(Directory!{col["Co-resident Count"]}2:{col["Co-resident Count"]}{last},">0")'),
        ("Entries with a coworker", f'=COUNTIF(Directory!{col["Coworker Count"]}2:{col["Coworker Count"]}{last},">0")'),
        ("Entries sharing a surname", f'=COUNTIF(Directory!{col["Same-Surname Count"]}2:{col["Same-Surname Count"]}{last},">0")'),
    ]
    r = 3
    ws2.cell(row=r, column=1, value="Metric").font = Font(name="Arial", bold=True)
    ws2.cell(row=r, column=2, value="Value").font = Font(name="Arial", bold=True)
    r += 1
    for label, formula in metrics:
        ws2.cell(row=r, column=1, value=label).font = Font(name="Arial", size=11)
        ws2.cell(row=r, column=2, value=formula).font = Font(name="Arial", size=11)
        r += 1
    ws2.column_dimensions["A"].width = 40
    ws2.column_dimensions["B"].width = 14

    r += 2
    ws2.cell(row=r, column=1, value="Notes").font = Font(name="Arial", bold=True, size=12)
    r += 1
    for note in [
        "Extracted by an automated pipeline (OpenCV layout detection, Google Gemini transcription and field parsing).",
        "'Marker as printed' reproduces the racial or ethnic identifier exactly as printed, blank where the directory printed none. Nothing is inferred or standardised; (c), (col) and (Chinese) are kept distinct.",
        "'Needs review' is a heuristic flag, split into two independent checks in 'Fragment flag' and 'Duplicate flag'. A fragment looks like a partial listing (mid-word start, surname with nothing after it) -- close to deterministic. A duplicate is near-identical full text to the entry right before it on the same page, at a conservative 0.92 similarity threshold chosen because this directory legitimately contains many real, different, similarly-worded neighbours (siblings, common surnames at a shared address). Both are a prompt to check the scan, not proof of an error.",
        "Linked Business / Co-resident / Coworker / Same-Surname are heuristic text matches. Spot-check before relying on them, especially for common surnames.",
        "Relationships are computed within an edition only; matches across directory years are not detected.",
    ]:
        ws2.cell(row=r, column=1, value="- " + note).font = Font(name="Arial", size=10, italic=True, color="5B6B85")
        ws2.merge_cells(start_row=r, start_column=1, end_row=r, end_column=6)
        ws2.row_dimensions[r].height = 30
        r += 1

    output_path.parent.mkdir(parents=True, exist_ok=True)
    wb.save(output_path)
    return output_path

## 11. Run it

If either output file is open in Excel or your browser, close it first or this will fail with a
permission error.

In [11]:
print("Loading Step 5 output...")
df = load_all()
print(f"  total {len(df)} entries, {df['source_page'].nunique()} pages\n")
print("Computing relationships...")
entries = compute_relationships(df)
print("Flagging entries that need review...")
entries = flag_quality(entries)
entries = add_derived(entries)
n_flag = sum(1 for e in entries if e["flag"])
n_frag = sum(1 for e in entries if e["fragment_flag"])
n_dup = sum(1 for e in entries if e["dup_flag"])
print(f"  {n_flag} of {len(entries)} flagged ({n_flag/len(entries)*100:.1f}%) "
      f"-- {n_frag} fragment, {n_dup} possible duplicate\n")
print("Writing outputs...")
h = build_html(entries, DASHBOARD_DIR / "Directory_Dashboard.html")
x = build_xlsx(entries, DASHBOARD_DIR / "Directory_Dashboard.xlsx")
print(f"  {h.resolve()}\n  {x.resolve()}")

Loading Step 5 output...
  1900-1901: 5621 entries
  total 5621 entries, 75 pages

Computing relationships...
Flagging entries that need review...
  159 of 5621 flagged (2.8%) -- 113 fragment, 46 possible duplicate

Writing outputs...
  C:\Users\ABHIRAMI.K\Downloads\dashboard_output\Directory_Dashboard.html
  C:\Users\ABHIRAMI.K\Downloads\dashboard_output\Directory_Dashboard.xlsx
